# 30 · E1 RAWD (Write-After-Read-Delay) · 真器件

> ⚠️ **执行前必读**
>
> 此 notebook 真连 B1500，接 pFeFET 器件。
>
> **前置确认**：
> - [ ] 23_wgfmu_wakeup_realdevice PASS (baseline 电流已知)
> - [ ] 器件已通过 wakeup 稳定化
> - [ ] GATE → CH201 (RSU), DRAIN → CH202 (RSU)
> - [ ] WGFMU_DLL_PATH 已设或 dll 在 System32

**目的 (H1/H2/H3)**：用写后延迟 t_delay 将 Qocc 快速电荷捕获、Ppre 慢弛豫、Nit 损伤分离。

序列：Reset(0V,1ms) → Write(±5V,100μs) → wait(t_delay) → 3pt-Read(Vg={-0.2,0,+0.2}V, Vd=0.05V, 5μs each)

测试人：**yhzang**

In [ ]:
import sys, os, time, random, datetime
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fefetlab.measurements.wgfmu import (
    ensure_wgfmu_dll_path,
    autodetect_visa_addr,
    autodetect_wgfmu_chan,
    RealWgfmuBackend,
)

print("Python:", sys.version.split()[0])
print("project root:", ROOT)

In [ ]:
dll = ensure_wgfmu_dll_path()
print(f"✅ wgfmu.dll: {dll}")

VISA_ADDR = autodetect_visa_addr("B1500")
print(f"✅ B1500 VISA addr: {VISA_ADDR}")

backend = RealWgfmuBackend()
backend.load()
backend.open_session(VISA_ADDR)
backend.set_timeout(30.0)
channel_ids = backend.get_channel_ids()
print(f"detected WGFMU channels: {channel_ids}")

GATE_CH  = autodetect_wgfmu_chan(backend, prefer=201)
DRAIN_CH = 202
assert DRAIN_CH in channel_ids, f"CH{DRAIN_CH} not found in {channel_ids}"
print(f"✅ gate CH={GATE_CH}, drain CH={DRAIN_CH}")

backend.close_session()

In [ ]:
# ── USER-EDITABLE PARAMETERS ──────────────────────────────────
QUICK_MODE  = True        # True: 5 delays × 3 reps (~5 min); False: 17 × 10

DEVICE_ID   = "dev001"    # DUT label
GEOMETRY    = "W5L10"     # gate W×L in μm

V_ERS       = +5.0        # V, ERS gate amplitude
V_PGM       = -5.0        # V, PGM gate amplitude
T_WRITE     = 100e-6      # s, write pulse width
T_RISE_FALL = 100e-9      # s, rise/fall time
T_RESET     = 1e-3        # s, reset hold at 0V before each shot

VG_READS    = [-0.2, 0.0, +0.2]   # V, 3-point read gate voltages
VD_READ     = 0.05                 # V, drain bias during read
T_READ      = 5e-6                 # s, read pulse width
N_PTS_READ  = 5                    # WGFMU sample points per read pulse

DELAYS_QUICK = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
DELAYS_FULL  = [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3,
                1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0, 10.0, 30.0, 100.0]

N_REPS_QUICK = 3
N_REPS_FULL  = 10
RANDOM_SEED  = 42
# ── END PARAMETERS ────────────────────────────────────────────

delays = DELAYS_QUICK if QUICK_MODE else DELAYS_FULL
N_REPS = N_REPS_QUICK if QUICK_MODE else N_REPS_FULL
TAG    = "quick" if QUICK_MODE else "full"

t_est = sum(d + T_RESET + T_WRITE + 3*(T_READ + 2*T_RISE_FALL)
            for d in delays) * N_REPS * 2
print(f"Mode : {TAG}")
print(f"Shots: {len(delays)} delays × {N_REPS} reps × 2 states = {len(delays)*N_REPS*2}")
print(f"Est. : {t_est:.1f} s  ({t_est/60:.1f} min)")

In [ ]:
def _run_shot(
    backend,
    gate_ch: int,
    drain_ch: int,
    v_write: float,
    t_write: float,
    t_rise_fall: float,
    t_reset: float,
    t_delay: float,
    vg_reads,
    vd_read: float,
    t_read: float,
    n_pts: int = 5,
) -> list:
    """Single write-delay-read shot on gate+drain channels.

    Returns list of dicts (one per Vg_read):
      Vg_read_V, Vd_read_V, Id_mean_A, Id_std_A, Ig_mean_A, n_d, n_g
    """
    T_GAP  = 100e-9
    GUARD  = 200e-9
    t_delay = max(t_delay, 200e-9)   # floor to avoid zero-width vectors

    backend.clear()

    # Offline: gate pattern
    backend.create_pattern("gp", 0.0)
    backend.add_vector("gp", t_reset,    0.0)       # reset hold
    backend.add_vector("gp", t_rise_fall, v_write)  # rise to write
    backend.add_vector("gp", t_write,    v_write)   # write hold
    backend.add_vector("gp", t_rise_fall, 0.0)      # fall
    backend.add_vector("gp", t_delay,    0.0)       # delay

    # Offline: drain pattern (0V through reset+write+delay, then read bias)
    t_pre = t_reset + t_rise_fall + t_write + t_rise_fall + t_delay
    backend.create_pattern("dp", 0.0)
    backend.add_vector("dp", t_pre, 0.0)  # hold 0V through entire pre-read phase

    # Read pulse vectors and measure events
    meas_win  = t_read - GUARD
    interval  = meas_win / max(n_pts, 1)
    average   = interval * 0.8

    t_cursor = t_pre
    read_t0s = []
    for i, vg in enumerate(vg_reads):
        t_cursor += t_rise_fall
        read_t0s.append(t_cursor)
        t_cursor += t_read
        t_cursor += t_rise_fall

        backend.add_vector("gp", t_rise_fall, float(vg))
        backend.add_vector("gp", t_read,       float(vg))
        backend.add_vector("gp", t_rise_fall,  0.0)

        backend.add_vector("dp", t_rise_fall, vd_read)
        backend.add_vector("dp", t_read,       vd_read)
        backend.add_vector("dp", t_rise_fall,  0.0)

        if i < len(vg_reads) - 1:
            t_cursor += T_GAP
            backend.add_vector("gp", T_GAP, 0.0)
            backend.add_vector("dp", T_GAP, 0.0)

    for i in range(len(vg_reads)):
        t_ev = read_t0s[i] + GUARD
        backend.set_measure_event("gp", f"g{i}", t_ev, n_pts, interval, average, "averaged")
        backend.set_measure_event("dp", f"d{i}", t_ev, n_pts, interval, average, "averaged")

    backend.add_sequence(gate_ch,  "gp", 1)
    backend.add_sequence(drain_ch, "dp", 1)

    # Online: channel setup
    backend.initialize()
    for ch in [gate_ch, drain_ch]:
        backend.set_operation_mode(ch, "FASTIV")
        backend.set_force_voltage_range(ch, "AUTO")
        backend.set_measure_enabled(ch, True)
        backend.set_measure_mode(ch, "CURRENT")
        backend.set_measure_current_range(ch, "1MA")
    backend.connect(gate_ch)
    backend.connect(drain_ch)

    backend.execute()
    backend.wait_until_completed()

    g_df = backend.get_measure_values(gate_ch)
    d_df = backend.get_measure_values(drain_ch)
    backend.disconnect(gate_ch)
    backend.disconnect(drain_ch)

    g_t = g_df["time_s"].values;  g_v = g_df["value"].values
    d_t = d_df["time_s"].values;  d_v = d_df["value"].values

    results = []
    for i, vg in enumerate(vg_reads):
        t0 = read_t0s[i] + GUARD
        t1 = t0 + meas_win
        dm = (d_t >= t0) & (d_t <= t1)
        gm = (g_t >= t0) & (g_t <= t1)
        d_sub, g_sub = d_v[dm], g_v[gm]
        results.append(dict(
            Vg_read_V=float(vg), Vd_read_V=vd_read,
            Id_mean_A = float(np.nanmean(d_sub)) if len(d_sub) > 0 else float("nan"),
            Id_std_A  = float(np.nanstd(d_sub))  if len(d_sub) > 1 else float("nan"),
            Ig_mean_A = float(np.nanmean(g_sub)) if len(g_sub) > 0 else float("nan"),
            n_d=int(len(d_sub)), n_g=int(len(g_sub)),
        ))
    return results


print("✅ _run_shot() defined")

In [ ]:
ts_start = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir  = ROOT / "runs" / f"{ts_start}_E1_rawd_{TAG}"
run_dir.mkdir(parents=True, exist_ok=True)
out_csv  = run_dir / "rawd_results.csv"
print(f"Output dir : {run_dir}")
print(f"Output CSV : {out_csv}")

random.seed(RANDOM_SEED)
delays_run = list(delays)
random.shuffle(delays_run)
print(f"Shuffled delay order: {[f'{d:.1e}' for d in delays_run]}")

In [ ]:
all_rows = []
seq_id   = 0

backend.open_session(VISA_ADDR)
t_exp0 = time.time()
try:
    total_shots = len(delays_run) * N_REPS * 2
    print(f"Starting E1 RAWD: {total_shots} shots")

    for delay_s in delays_run:
        shot_timeout = max(30.0, delay_s * 3 + 10.0)
        backend.set_timeout(shot_timeout)

        for rep in range(N_REPS):
            for state, v_write in [("ERS", V_ERS), ("PGM", V_PGM)]:
                ts_iso = datetime.datetime.now().isoformat(timespec="seconds")
                t_sh   = time.time()
                note   = ""
                try:
                    rr = _run_shot(
                        backend, GATE_CH, DRAIN_CH,
                        v_write=v_write, t_write=T_WRITE,
                        t_rise_fall=T_RISE_FALL, t_reset=T_RESET,
                        t_delay=delay_s, vg_reads=VG_READS,
                        vd_read=VD_READ, t_read=T_READ, n_pts=N_PTS_READ,
                    )
                except Exception as exc:
                    note = f"ERR:{exc}"
                    print(f"  !! {state} d={delay_s:.1e} rep={rep}: {exc}")
                    rr = [dict(Vg_read_V=vg, Vd_read_V=VD_READ,
                               Id_mean_A=float("nan"), Id_std_A=float("nan"),
                               Ig_mean_A=float("nan"), n_d=0, n_g=0)
                          for vg in VG_READS]
                    # recover: reopen session
                    try:
                        backend.disconnect(GATE_CH)
                    except Exception:
                        pass
                    try:
                        backend.disconnect(DRAIN_CH)
                    except Exception:
                        pass
                    try:
                        backend.close_session()
                    except Exception:
                        pass
                    time.sleep(1.0)
                    backend.open_session(VISA_ADDR)
                    backend.set_timeout(shot_timeout)

                for r in rr:
                    all_rows.append(dict(
                        timestamp_iso=ts_iso, device_id=DEVICE_ID,
                        geometry=GEOMETRY, sequence_id=seq_id,
                        repeat_index=rep, state_target=state,
                        delay_s=delay_s,
                        Vg_read_V=r["Vg_read_V"], Vd_read_V=r["Vd_read_V"],
                        Id_mean_A=r["Id_mean_A"], Id_std_A=r["Id_std_A"],
                        Ig_mean_A=r["Ig_mean_A"], note=note,
                    ))
                seq_id += 1

                vg0_Id = rr[1]["Id_mean_A"]  # Vg=0V
                print(f"  {state:3s} d={delay_s:.1e}s rep={rep:2d} | "
                      f"Id(0V)={vg0_Id*1e9 if not np.isnan(vg0_Id) else float('nan'):.1f}nA | "
                      f"{time.time()-t_sh:.2f}s")

        # Incremental save after each delay point
        pd.DataFrame(all_rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

    total_t = time.time() - t_exp0
    print(f"\n✅ Done. {seq_id} shots in {total_t:.1f}s ({total_t/60:.1f} min)")
    print(f"Saved: {out_csv}")

except Exception as exc:
    print(f"\n❌ Experiment failed: {exc}")
    raise
finally:
    for ch in [GATE_CH, DRAIN_CH]:
        try:
            backend.disconnect(ch)
        except Exception:
            pass
    backend.close_session()

df = pd.DataFrame(all_rows)
print(df.shape, df.dtypes.to_dict())

In [ ]:
df = pd.read_csv(out_csv)
vg_uniq = sorted(df["Vg_read_V"].unique())

fig, axes = plt.subplots(1, len(vg_uniq), figsize=(5*len(vg_uniq), 5), sharey=False)
if len(vg_uniq) == 1:
    axes = [axes]

colors = {"ERS": "#1f77b4", "PGM": "#d62728"}
for ax, vg in zip(axes, vg_uniq):
    for state, color in colors.items():
        sub  = df[(df["Vg_read_V"] == vg) & (df["state_target"] == state)]
        grp  = sub.groupby("delay_s")["Id_mean_A"].agg(["mean", "std"]).reset_index()
        ax.errorbar(grp["delay_s"] * 1e3, grp["mean"] * 1e9,
                    yerr=grp["std"] * 1e9, fmt="o-",
                    color=color, label=state, ms=5, capsize=3)
    ax.set_xscale("log")
    ax.set_xlabel("delay (ms)")
    ax.set_ylabel("Id (nA)")
    ax.set_title(f"Vg_read = {vg:+.1f}V")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle(f"E1 RAWD – Id vs delay | {DEVICE_ID} {GEOMETRY}")
plt.tight_layout()
plt.savefig(run_dir / "rawd_id_vs_delay.png", dpi=150, bbox_inches="tight")
plt.show()

# MW = Id(ERS) - Id(PGM) at each Vg_read
fig2, axes2 = plt.subplots(1, len(vg_uniq), figsize=(5*len(vg_uniq), 4), sharey=False)
if len(vg_uniq) == 1:
    axes2 = [axes2]
for ax, vg in zip(axes2, vg_uniq):
    ers_g = df[(df["Vg_read_V"] == vg) & (df["state_target"] == "ERS")].groupby("delay_s")["Id_mean_A"].mean()
    pgm_g = df[(df["Vg_read_V"] == vg) & (df["state_target"] == "PGM")].groupby("delay_s")["Id_mean_A"].mean()
    mw    = (ers_g - pgm_g).reset_index()
    mw.columns = ["delay_s", "MW_A"]
    ax.plot(mw["delay_s"] * 1e3, mw["MW_A"] * 1e9, "s-", color="#2ca02c", ms=6)
    ax.axhline(0, color="gray", lw=0.8, ls="--")
    ax.set_xscale("log")
    ax.set_xlabel("delay (ms)")
    ax.set_ylabel("MW = Id(ERS)-Id(PGM)  (nA)")
    ax.set_title(f"MW vs delay | Vg={vg:+.1f}V")
    ax.grid(alpha=0.3)
    pos = int((mw["MW_A"] > 0).sum())
    print(f"Vg={vg:+.1f}V: MW>0 at {pos}/{len(mw)} delay points")

plt.suptitle(f"E1 RAWD – Memory Window | {DEVICE_ID} {GEOMETRY}")
plt.tight_layout()
plt.savefig(run_dir / "rawd_mw_vs_delay.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nAll plots saved to {run_dir}")

## 通过标准

- [ ] 无 dll/VISA/通道错误
- [ ] CSV 行数 = len(delays) × N_REPS × 2 states × 3 Vg_reads
- [ ] Id(ERS) > Id(PGM) at least at short delay (< 1ms)
- [ ] MW 随 delay 变化趋势清晰（正或负）
- [ ] Ig_mean < 1 µA (无明显栅漏)

**结果记录**：MW 正负转折点 delay = ___ ms，最大 |MW| = ___ nA at Vg = ___ V